In [22]:
import torch
import numpy as np


class WeaklyConvexProblem:
    """
    Defines the objective function phi(x) = f(x) + r(x).
    Based on Example 2.1 (Phase Retrieval) or simple |x^2 - 1|.
    f_objective: The stochastic loss function f(x)
    """

    def __init__(self, f_objective, r_objective, model_gen, rho = 2.0):
        self.f = f_objective
        self.r = r_objective
        self.get_model = model_gen
        self.rho = rho # Used to guide the Solver's beta_t

    def objective(self, x):
        return self.f(x) + self.r(x)

In [8]:
class ModelBasedSolver:
    """
    Implements Algorithm 1.2: Stochastic Model-Based Minimization .
    """
    def __init__(self, problem, x_init, T, gamma=0.1):
        self.prob = problem
        self.x = x_init.clone().detach()
        self.T = T
        self.gamma = gamma # Step-size parameter
        self.history = []
        
    def get_beta(self, t):
        # The paper requires beta_t > rho 
        # We add a small constant (e.g., 0.1) to ensure strict strong convexity
        rho_hat = self.prob.rho + 0.1 
        return rho_hat + (1.0 / self.gamma) * np.sqrt(t + 1.0)

    def run(self):
        
        for t in range(self.T):            
            beta_t = self.get_beta(t)
            x_t = self.x.clone().detach()
            g_xt = self.prob.get_model(x_t)
            
            # SUBPROBLEM: x_{t+1} = argmin { f_xt(y) + (1/2*alpha) * ||y-x_t||^2 }
            y = x_t.clone().requires_grad_(True)
            inner_opt = torch.optim.LBFGS([y], lr=1, max_iter=20)

            def closure():                
                inner_opt.zero_grad()
                model_val = g_xt(y) + self.prob.r(y) # Add the regularizer r(y)
                penalty = (beta_t / 2) * torch.norm(y - x_t)**2
                loss = model_val + penalty
                loss.backward()
                return loss

            inner_opt.step(closure)
            self.x = y.detach().clone()
            
            if t % 50 == 0:
                print(f"Iter {t}: x = {self.x.item():.4f}, f(x) = {self.prob.objective(self.x).item():.4f}")

        return self.x

In [3]:
def l1_regularizer(x):
    return 0.5 * torch.norm(x, p=1) # Encourages sparsity

In [24]:
def abs_parabola(x):
    return torch.abs(x**2 - 1.0)

def abs_parabola_model(x_t):
    c_xt = x_t**2 - 1.0
    jc_xt = 2.0 * x_t
    
    def model(y):
        # h(c(x_t) + Jc(x_t)(y - x_t))
        linearized_inner = c_xt + jc_xt * (y - x_t)
        return torch.abs(linearized_inner)
    return model

In [25]:
problem = WeaklyConvexProblem(f_objective=abs_parabola, r_objective=l1_regularizer, model_gen=abs_parabola_model)
solver_max = ModelBasedSolver(problem, x_init=torch.tensor([10.0]), T=500, gamma=0.5)
final_x_max = solver_max.run()

print(f"Final x for Pointwise Max: {final_x_max.item():.4f}")

Iter 0: x = 5.0000, f(x) = 26.5000
Iter 50: x = 1.0290, f(x) = 0.5734
Iter 100: x = 1.0047, f(x) = 0.5117
Iter 150: x = 1.0248, f(x) = 0.5626
Iter 200: x = 1.0205, f(x) = 0.5516
Iter 250: x = 1.0172, f(x) = 0.5432
Iter 300: x = 1.0175, f(x) = 0.5441
Iter 350: x = 1.0069, f(x) = 0.5173
Iter 400: x = 1.0159, f(x) = 0.5399
Iter 450: x = 1.0122, f(x) = 0.5306
Final x for Pointwise Max: 1.0108


In [31]:
def max_parabola_phi(x):
    """The 'True' Objective: max(x^2, (x-4)^2)"""
    f1 = x ** 2
    f2 = (x - 4.0) ** 2
    return torch.max(f1, f2)


def max_parabola_model_gen(x_t):
    """
    Returns the Stochastic Model for the Pointwise Max.
    Based on Example 2.6 and Section 4.2[cite: 984].
    """
    # Detach constants for the inner loop
    x_t_val = x_t.detach()
    f1_val = x_t_val ** 2
    f2_val = (x_t_val - 4.0) ** 2

    grad1 = 2 * x_t_val
    grad2 = 2 * (x_t_val - 4.0)

    def model(y):
        # Linearize both components inside the max
        l1 = f1_val + grad1 * (y - x_t_val)
        l2 = f2_val + grad2 * (y - x_t_val)
        return torch.max(l1, l2)

    return model

def elastic_net_regularizer(x, l1_ratio=0.5, mu=0.01):
    """
    Combines L1 and L2 regularization. 
    Matches the 'regularized population risk' framework[cite: 13, 20].
    """
    l1_term = l1_ratio * torch.norm(x, p=1)
    l2_term = (1 - l1_ratio) * 0.5 * torch.norm(x, p=2)**2
    # Ensure the result is a tensor attached to the same device as x
    return mu * (l1_term + l2_term)

In [32]:
# Initialize Problem
problem = WeaklyConvexProblem(
    f_objective=max_parabola_phi, 
    r_objective=elastic_net_regularizer, 
    model_gen=max_parabola_model_gen,
    rho=2.0 # Curvature of x^2 is 2
)

# Run Solver
solver = ModelBasedSolver(problem, x_init=torch.tensor([10.0]), T=800, gamma=0.5)
final_x = solver.run()

print(f"Final x: {final_x.item():.4f}")

Iter 0: x = 5.1145, f(x) = 26.2490
Iter 50: x = 1.9677, f(x) = 4.1499
Iter 100: x = 1.9863, f(x) = 4.0747
Iter 150: x = 2.0350, f(x) = 4.1617
Iter 200: x = 1.9598, f(x) = 4.1820
Iter 250: x = 1.9304, f(x) = 4.3020
Iter 300: x = 1.9458, f(x) = 4.2390
Iter 350: x = 2.0294, f(x) = 4.1390
Iter 400: x = 2.0063, f(x) = 4.0454
Iter 450: x = 1.9550, f(x) = 4.2015
Iter 500: x = 2.0258, f(x) = 4.1243
Iter 550: x = 1.9514, f(x) = 4.2160
Iter 600: x = 2.0394, f(x) = 4.1799
Iter 650: x = 1.9472, f(x) = 4.2330
Iter 700: x = 2.0208, f(x) = 4.1038
Iter 750: x = 2.0418, f(x) = 4.1897
Final x: 2.0213
